In [1]:
import os
import pathlib

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import tensorflow as tf

from keras import layers
from keras import models
from IPython import display


This block defines a function to load a Keras model from a specified filepath and then uses it to load `model_export.keras`.

It then prints a detailed summary of the model, including the number of layers, parameters per layer, and the total trainable parameters.

The code prints the total number of layers and iterates through each one to display its name, type and the number of parameters it contains. It also calculated the total number of paramters across all layers, given an overview of the model's size and complexity.


In [2]:
import sys

import numpy as np
from keras import Model
from keras import Layer

def import_model(filepath: str) -> Model:
    """Load model from file"""
    model: Model = models.load_model(filepath)
    return model

# Load the pre-trained Keras model from the specified file
model = import_model('model_export.keras')

# Print a summary of the model's architecture and parameters
print("Model Summary:")
print(f"Total layers: {len(model.layers)}")
total_params = 0
# Iterate through each layer to display its name, type, and parameter count
for i, layer in enumerate(model.layers):
    print(f"Layer {i+1}: {layer.name} ({layer.__class__.__name__}) - Parameters: {layer.count_params()}")
    total_params += layer.count_params()
print(f"Total trainable parameters: {total_params}")

# Also print the full summary as it might contain more details (e.g., output shapes)
print("\nFull model.summary():")
model.summary()

Model Summary:
Total layers: 5
Layer 1: embedding_1 (Embedding) - Parameters: 804496
Layer 2: conv1d_1 (Conv1D) - Parameters: 6272
Layer 3: global_max_pooling1d_1 (GlobalMaxPooling1D) - Parameters: 0
Layer 4: dense_2 (Dense) - Parameters: 8256
Layer 5: dense_3 (Dense) - Parameters: 3268265
Total trainable parameters: 4087289

Full model.summary():


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ (None, 100, 16)        │       804,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ (None, 100, 128)       │         6,272 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_max_pooling1d_1          │ (None, 128)            │             0 │
│ (GlobalMaxPooling1D)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 50281)          │     3,268,265 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 12,261,869 (46.78 MB)

 Trainable params: 4,087,289 (15.59 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 8,174,580 (31.18 MB)

These utility functions are essential for the manual verification process. `get_layer_weights` retrieves the weights and biases of a specific layer. `get_weights_shape` is a helper for `get_layer_weights`. `get_reference_layer` is particularly clever, as it allows us to isolate a specific layer by creating two `Sequential` models: one containing layers *before* the target, and one containing layers *after*. A `DummyLayer` is used to handle cases where there are no layers before or after, ensuring the `Sequential` models can be built.



In [3]:
def get_layer_weights(layer: str, model: Model) -> list[np.ndarray]:
    """Get layer weights from given model"""
    return model.get_layer(layer).get_weights()

def get_weights_shape(layer_weights: list[np.ndarray]):
    """Get the shapes of the given layer weights"""
    tmp = []
    for i in layer_weights:
        tmp.append(i.shape)
    return tmp

class DummyLayer(Layer):
    """A placeholder layer that passes its inputs directly, useful for model construction."""
    def __init__(self):
        super(DummyLayer, self).__init__()

    def call(self, inputs):
        return inputs

def get_reference_layer(layer_name: str, model: Model):
    """
    Splits a Keras model into three parts relative to a specified layer:
    1. A Sequential model of layers before the specified layer.
    2. The specified layer itself.
    3. A Sequential model of layers after the specified layer.
    Dummy layers are used if there are no layers before/after.
    """
    model_start = models.Sequential()
    selected_layer: Layer
    model_end = models.Sequential()

    start_collecting_end_layers = False
    for l in model.layers:
        if l.name == layer_name:
            start_collecting_end_layers = True
            selected_layer = l
            continue
        if not start_collecting_end_layers:
            model_start.add(l)
        else:
            model_end.add(l)


    if not len(model_start.layers):
        model_start.add(DummyLayer())
    if not len(model_end.layers):
        model_end.add(DummyLayer())

    model_start.build(model.input_shape)
    model_end.build(selected_layer.output.shape)

    return (model_start, selected_layer, model_end)

This block demonstrates how to use OpenAI's `tiktoken` library for text tokenization. It first installs the library, then defines a sample text, encodes it into tokens using the 'p50k_base' encoding, and finally prints the tokens and the vocabulary size.



In [4]:
!pip install tiktoken
import tiktoken

text = "This is an input text sequence to the model"

# Get the 'p50k_base' encoding, which is a common encoding for GPT-2/3 models
enc = tiktoken.get_encoding("p50k_base")

# Encode the text into a list of integer tokens
tokens = enc.encode(text)

# Print the resulting tokens
print(tokens)
# Get and print the vocabulary size of the encoder
vocab_size = enc.n_vocab
print(vocab_size)

[1212, 318, 281, 5128, 2420, 8379, 284, 262, 2746]
50281


This simple block demonstrates the reverse process of tokenization: decoding a list of tokens back into a human-readable string using the same `tiktoken` encoder. This is important for understanding the model's generated output.


In [5]:
# Decode the list of tokens back into human-readable text using the same encoder
decoded = enc.decode(tokens)
# Print the decoded text to verify the tokenization and decoding process
print(decoded)


This is an input text sequence to the model


This block implements a text generation process that uses the trained keras model to predict the next token in a sequence. Starting form an initial set of tokens, the model generates 100 new tokens one step at a time.

At each step, the current token sequence is padded to a fixed length using `pad_sequences` to match the model’s expected input size. The model then predicts a probability distribution over possible next tokens.

The `sample_with_temperature `function is used to select the next token from this distribution. By applying temperature scaling, it controls randomness—allowing the output to be either more deterministic or more diverse.

After each prediction, the new token is appended to the sequence, and only the most recent 30 tokens are kept (a sliding window) to maintain a consistent input size for the model.



In [6]:
from tensorflow.keras.preprocessing.sequence import pad_sequences
import tiktoken

text = "This is an input text sequence to the model"
enc = tiktoken.get_encoding("p50k_base")
tokens = enc.encode(text)

def sample_with_temperature(pred: np.ndarray, temperature: float = 1.0) -> int:
    """
    Samples a token from prediction probabilities, adjusted by temperature.
    Higher temperature means more diverse (random) samples.
    """
    pred = np.log(pred[0] + 1e-9) / temperature
    pred = np.exp(pred)
    pred = pred / np.sum(pred)

    return np.random.choice(len(pred), p=pred)


def makePredictions(initial_tokens: list[int], T: int) -> list[int]:
    """
    Generates a sequence of T tokens using the loaded model.
    """
    tokens_sequence = list(initial_tokens)   # Ensure it's a mutable list
    generated_outputs = []

    for i in range(T):

        # Pad the current token sequence to a fixed length (e.g., 30) for model input
        padded_input = pad_sequences([tokens_sequence], maxlen=30, padding="pre")
        padded_input = np.array(padded_input)   # Ensure it's a numpy array

        # Make a prediction with the model (verbose=0 to suppress output)
        pred = model.predict(padded_input, verbose=0)

        # Select the next token using temperature-based sampling
        next_token = sample_with_temperature(pred, temperature=1)

        # Store the generated token
        generated_outputs.append(next_token)

        # Add the predicted token to the sequence for the next prediction round
        tokens_sequence.append(next_token)

        # Keep only the last 30 tokens to maintain fixed input size (sliding window)
        tokens_sequence = tokens_sequence[-30:]

    return generated_outputs

# Generate 100 predicted tokens using the initial input tokens
predictedTokens = makePredictions(tokens, 100)
print("Predicted Tokens:", predictedTokens)

# Decode the generated tokens back into human-readable text
decoded = enc.decode(predictedTokens)
print("\nDecoded Generated Text:", decoded)

Predicted Tokens: [1181, 7686, 4253, 198, 4833, 2084, 284, 35, 447, 251, 16, 1429, 19871, 11, 11, 284, 198, 635, 1802, 22238, 5832, 282, 8601, 636, 198, 1100, 2300, 318, 1711, 8776, 39938, 35582, 281, 5794, 17848, 329, 392, 1188, 867, 4327, 850, 250, 82, 16990, 24255, 6783, 2683, 1620, 345, 3155, 262, 7174, 7824, 561, 505, 13, 761, 69, 263, 923, 534, 1998, 1339, 3227, 7987, 5059, 3341, 515, 25912, 326, 42389, 616, 12556, 2138, 2925, 13, 363, 2931, 11379, 2649, 6809, 2536, 668, 261, 268, 13, 5310, 2884, 576, 11745, 26753, 21004, 910, 1178, 7679, 21754, 477, 5703, 540, 326]

Decoded Generated Text:  state networks categ
 smaller ago toD”1 process manif,, to
 also 100ifiyoualmaking part
 read matter is hour trained decoding sequential an format pixels forand val many tend sub�s VMdrivinggent questions perform you couple thecalled API wouldone. needfer start your experience case production regularly driving systemsated fi that leveraging my Early rather goes.ag09foldification participantss

This block verifies the `embedding_1` layer by comparing its Keras output with a manual calculation. The input tokens are first padded to a fixed length to match the model’s expected input. The embedding layer’s weights are then retrieved—each row of the weight matrix corresponds to a token vector.

The Keras output is obtained by passing the padded input through the layer. The manual output is computed by directly indexing the embedding matrix with the token IDs. Finally, the two outputs are compared using `np.allclose` and the maximum absolute difference is printed, confirming that the embedding layer simply performs a token-to-vector lookup.

In [7]:
import numpy as np
from tensorflow.keras.preprocessing.sequence import pad_sequences # ensure this is imported
import math

# Prepare input (same as before for consistency in verification)
# This 'padded' input will be used for all layer verifications
padded = pad_sequences([tokens], maxlen=30, padding="pre")
padded = np.array(padded)

print(model.summary()) # Display model summary again for context


# Manual implementation and verification for EMBEDDING_1 layer
print("\n--- Verifying Embedding_1 Layer ---")

# Get model parts relevant to embedding_1 using helper function
# 'start_model_embedding1' will be DummyLayer as it's the first layer
# 'layer_embedding1' is the 'embedding_1' layer itself
(start_model_embedding1, layer_embedding1, end_model_embedding1) = get_reference_layer("embedding_1", model)

# Getting the Keras output for embedding_1 by passing the padded input through it
k_embedding1_output = layer_embedding1(padded).numpy()

# Getting the embedding matrix (weights) for embedding_1
w_embedding1 = get_layer_weights("embedding_1", model)[0]
print(f"Embedding_1 weights shape: {w_embedding1.shape}")

# Manual calculation for embedding_1 layer: perform lookup
# The padded input contains token IDs, which are used as indices into the embedding matrix
manual_embedding1_output = w_embedding1[padded]

# Compare with Keras output for embedding_1
print("\nShape of Keras Embedding_1 output:", k_embedding1_output.shape)
print("Shape of manually calculated Embedding_1 output:", manual_embedding1_output.shape)
print("Are Embedding_1 outputs approximately equal?", np.allclose(k_embedding1_output, manual_embedding1_output, rtol=1e-5, atol=1e-8))
print("Max absolute difference for Embedding_1:", np.max(np.abs(k_embedding1_output - manual_embedding1_output)))

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ (None, 100, 16)        │       804,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ (None, 100, 128)       │         6,272 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_max_pooling1d_1          │ (None, 128)            │             0 │
│ (GlobalMaxPooling1D)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 50281)          │     3,268,265 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 12,261,869 (46.78 MB)

 Trainable params: 4,087,289 (15.59 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 8,174,580 (31.18 MB)

None

--- Verifying Embedding_1 Layer ---

Embedding_1 weights shape: (50281, 16)

Shape of Keras Embedding_1 output: (1, 30, 16)
Shape of manually calculated Embedding_1 output: (1, 30, 16)
Are Embedding_1 outputs approximately equal? True
Max absolute difference for Embedding_1: 0.0


This block manually verifies the `conv1d_1` layer by reproducing its computation step by step. It first gets the layer’s input (output of `embedding_1`), kernel, bias, and parameters like kernel size, stride, padding, and activation. The input is padded according to the layer’s padding type (`same`, `valid`, or `causal`).

The convolution is computed manually by sliding the kernel over the input, performing element-wise multiplication, summing, and adding the bias for each filter and timestep. The activation function (e.g., ReLU) is then applied. Finally, the manual output is compared with the Keras output using `np.allclose` and the maximum difference is printed, confirming that the layer behaves as expected.

In [8]:
# Manual implementation and verification for CONV1D_1 layer
print("\n--- Verifying Conv1D_1 Layer ---")

# Getting model parts relevant to conv1d_1 using our helper
(start_model_conv1d1, layer_conv1d1, end_model_conv1d1) = get_reference_layer("conv1d_1", model)

# Getting the Keras output for conv1d_1. The input to conv1d_1 is the output of embedding_1.
k_conv1d1_output = layer_conv1d1(start_model_conv1d1(padded)).numpy()

# Getting the input to conv1d_1, which is effectively the output of the embedding_1 layer
input_to_conv1d1 = start_model_conv1d1(padded).numpy()

# Getting weights (kernel) and bias for conv1d_1
w_conv1d1_kernel, b_conv1d1 = get_layer_weights("conv1d_1", model)
print(f"Conv1D_1 kernel shape: {w_conv1d1_kernel.shape}") # Expected: (kernel_size, input_dim, filters)
print(f"Conv1D_1 bias shape: {b_conv1d1.shape}")       # Expected: (filters,)

# Getting Conv1D parameters from the Keras layer object
conv1d1_obj = model.get_layer("conv1d_1")
kernel_size_conv1d1 = conv1d1_obj.kernel_size[0]
strides_conv1d1 = conv1d1_obj.strides[0]
padding_conv1d1 = conv1d1_obj.padding
activation_conv1d1 = conv1d1_obj.activation
print(f"Conv1D_1 params: kernel_size={kernel_size_conv1d1}, strides={strides_conv1d1}, padding='{padding_conv1d1}', activation='{activation_conv1d1.__name__}'")

# Manual calculation for Conv1D_1
batch_size, input_timesteps, input_dim = input_to_conv1d1.shape
filters = w_conv1d1_kernel.shape[2]

# Determine output timesteps and handle padding for 'same' mode
output_timesteps = 0 # Initialize
if padding_conv1d1 == 'valid':
    # 'valid' padding means no padding, output size shrinks
    output_timesteps = (input_timesteps - kernel_size_conv1d1) // strides_conv1d1 + 1
elif padding_conv1d1 == 'same':
    # 'same' padding attempts to keep output size the same as input size
    output_timesteps = math.ceil(input_timesteps / strides_conv1d1)

    # Calculate required padding amount
    if (input_timesteps % strides_conv1d1 == 0):
        pad_along_axis = max(kernel_size_conv1d1 - strides_conv1d1, 0)
    else:
        pad_along_axis = max(kernel_size_conv1d1 - (input_timesteps % strides_conv1d1), 0)

    pad_left = pad_along_axis // 2
    pad_right = pad_along_axis - pad_left

    # Apply padding to the input
    input_to_conv1d1_padded = np.pad(input_to_conv1d1, ((0,0), (pad_left, pad_right), (0,0)), mode='constant')

    # Update input_timesteps for the convolution loop to reflect padded dimensions
    input_timesteps = input_to_conv1d1_padded.shape[1]
    input_for_conv = input_to_conv1d1_padded
elif padding_conv1d1 == 'causal': # Add causal padding if needed for this specific model type
    # For causal, output[t] depends only on input[0:t]
    # Typically, pad_left = kernel_size - 1, pad_right = 0
    pad_left = kernel_size_conv1d1 - 1
    pad_right = 0
    input_to_conv1d1_padded = np.pad(input_to_conv1d1, ((0,0), (pad_left, pad_right), (0,0)), mode='constant')
    input_timesteps = input_to_conv1d1_padded.shape[1]
    input_for_conv = input_to_conv1d1_padded
    output_timesteps = (input_timesteps - kernel_size_conv1d1) // strides_conv1d1 + 1
else:
    raise ValueError(f"Unsupported padding: {padding_conv1d1}")

# If padding is not 'same', use the original input (or already calculated padded input for 'causal')
if padding_conv1d1 not in ['same', 'causal']:
    input_for_conv = input_to_conv1d1

# Initialize array to store pre-activation results
manual_conv1d1_pre_activation = np.zeros((batch_size, output_timesteps, filters))

# Loop over batch, output timesteps, and filters to perform convolution manually
for b in range(batch_size):
    for i in range(output_timesteps):
        # Determine the start and end indices for the convolution window in the input
        start_idx = i * strides_conv1d1
        end_idx = start_idx + kernel_size_conv1d1

        # Extract the slice of the input corresponding to the current window
        input_slice = input_for_conv[b, start_idx:end_idx, :]

        for f in range(filters):
            # Perform element-wise multiplication of input slice and kernel, then sum, and add bias
            # Kernel shape is (kernel_size, input_dim, filters)
            # input_slice shape is (kernel_size, input_dim)
            manual_conv1d1_pre_activation[b, i, f] = np.sum(input_slice * w_conv1d1_kernel[:, :, f]) + b_conv1d1[f]

# Apply activation function
manual_conv1d1_output = manual_conv1d1_pre_activation
if activation_conv1d1.__name__ == 'relu':
    manual_conv1d1_output = np.maximum(0, manual_conv1d1_pre_activation)
elif activation_conv1d1.__name__ != 'linear':
     print(f"Warning: Unhandled activation function for conv1d_1: {activation_conv1d1.__name__}. Assuming linear for comparison.")

# Compare with Keras output for conv1d_1
print("\nShape of Keras Conv1D_1 output:", k_conv1d1_output.shape)
print("Shape of manually calculated Conv1D_1 output:", manual_conv1d1_output.shape)
print("Are Conv1D_1 outputs approximately equal?", np.allclose(k_conv1d1_output, manual_conv1d1_output, rtol=1e-5, atol=1e-8))
print("Max absolute difference for Conv1D_1:", np.max(np.abs(k_conv1d1_output - manual_conv1d1_output)))


--- Verifying Conv1D_1 Layer ---
Conv1D_1 kernel shape: (3, 16, 128)
Conv1D_1 bias shape: (128,)
Conv1D_1 params: kernel_size=3, strides=1, padding='same', activation='relu'

Shape of Keras Conv1D_1 output: (1, 30, 128)
Shape of manually calculated Conv1D_1 output: (1, 30, 128)
Are Conv1D_1 outputs approximately equal? True
Max absolute difference for Conv1D_1: 4.76837158203125e-07


This block manually verifies the `global_max_pooling1d_1` layer. It first gets the input to the layer (output of `conv1d_1`) and computes the Keras output. The manual calculation is done by taking the maximum value across the sequence dimension for each feature (axis=1), which is exactly what global max pooling does. Finally, the manual output is compared with the Keras output using `np.allclose` and the maximum difference is printed, confirming that the layer behaves as expected.

In [9]:
# Manual implementation and verification for GLOBAL_MAX_POOLING1D_1 layer
print("\n--- Verifying GlobalMaxPooling1D_1 Layer ---")

# Getting model parts relevant to global_max_pooling1d_1 using helper
(start_model_gmp1d1, layer_gmp1d1, end_model_gmp1d1) = get_reference_layer("global_max_pooling1d_1", model)

# Getting the Keras output for global_max_pooling1d_1
k_gmp1d1_output = layer_gmp1d1(start_model_gmp1d1(padded)).numpy()

# Getting input to global_max_pooling1d_1 (which is the output of conv1d_1 from Keras pipeline)
input_to_gmp1d1 = start_model_gmp1d1(padded).numpy()

# Manual calculation for global_max_pooling1d_1 layer
# Global max pooling takes the maximum value over the entire sequence dimension (axis=1) for each feature
manual_gmp1d1_output = np.max(input_to_gmp1d1, axis=1)

# Compare with Keras output for global_max_pooling1d_1
print("\nShape of Keras GlobalMaxPooling1D_1 output:", k_gmp1d1_output.shape)
print("Shape of manually calculated GlobalMaxPooling1D_1 output:", manual_gmp1d1_output.shape)
print("Are GlobalMaxPooling1D_1 outputs approximately equal?", np.allclose(k_gmp1d1_output, manual_gmp1d1_output, rtol=1e-5, atol=1e-8))
print("Max absolute difference for GlobalMaxPooling1D_1:", np.max(np.abs(k_gmp1d1_output - manual_gmp1d1_output)))


--- Verifying GlobalMaxPooling1D_1 Layer ---

Shape of Keras GlobalMaxPooling1D_1 output: (1, 128)
Shape of manually calculated GlobalMaxPooling1D_1 output: (1, 128)
Are GlobalMaxPooling1D_1 outputs approximately equal? True
Max absolute difference for GlobalMaxPooling1D_1: 0.0


This block manually verifies the `dense_2` layer. It first gets the input to the layer (output of `global_max_pooling1d_1`) and retrieves the layer’s weights, bias, and activation function. The manual output is calculated by performing a matrix multiplication of the input with the weights, adding the bias, and then applying the activation function (e.g., softmax or ReLU). Finally, the manual output is compared with the Keras output using `np.allclose` and the maximum difference is printed, confirming the layer behaves as expected.

In [10]:
# Manual implementation and verification for DENSE_2 layer
print("\n--- Verifying Dense_2 Layer ---")

# Getting model parts relevant to dense_2 using helper
# 'start_model_dense2' represents layers before 'dense_2'
# 'layer_dense2' is the 'dense_2' layer itself
(start_model_dense2, layer_dense2, end_model_dense2) = get_reference_layer("dense_2", model)

# Get the Keras output for dense_2 (i.e., running 'start_model_dense2' then 'layer_dense2')
k_dense2_output = layer_dense2(start_model_dense2(padded)).numpy()

# Get input to dense_2 from the Keras pipeline (output of global_max_pooling1d_1)
input_to_dense2 = start_model_dense2(padded).numpy()

# Get weights (kernel) and bias for dense_2
w_dense2, b_dense2 = get_layer_weights("dense_2", model)
print(f"Dense_2 weights shape: {w_dense2.shape}")
print(f"Dense_2 bias shape: {b_dense2.shape}")

# Get the activation function of the dense_2 layer
dense2_activation = model.get_layer("dense_2").activation
print(f"Dense_2 activation function: {dense2_activation.__name__}")

# Manual calculation for dense_2 layer (pre-activation): input * weights + bias
manual_dense2_pre_activation = np.dot(input_to_dense2, w_dense2) + b_dense2

# Apply the activation function for dense_2
manual_dense2_output = manual_dense2_pre_activation # Default for 'linear' activation
if dense2_activation.__name__ == 'softmax':
    # Softmax for multi-class classification probabilities
    exp_values = np.exp(manual_dense2_pre_activation - np.max(manual_dense2_pre_activation, axis=-1, keepdims=True)) # Subtract max for numerical stability
    manual_dense2_output = exp_values / np.sum(exp_values, axis=-1, keepdims=True)
elif dense2_activation.__name__ == 'relu':
    # ReLU activation: max(0, x)
    manual_dense2_output = np.maximum(0, manual_dense2_pre_activation)
# Adding other common activations here if needed (e.g., tanh, sigmoid)
else:
    if dense2_activation.__name__ != 'linear': # Avoid warning for explicit linear activation
        print(f"Warning: Unhandled activation function for dense_2: {dense2_activation.__name__}. Assuming linear for comparison.")


# Compare with Keras output for dense_2
print("\nShape of Keras Dense_2 output:", k_dense2_output.shape)
print("Shape of manually calculated Dense_2 output:", manual_dense2_output.shape)
print("Are Dense_2 outputs approximately equal?", np.allclose(k_dense2_output, manual_dense2_output, rtol=1e-5, atol=1e-8))
print("Max absolute difference for Dense_2:", np.max(np.abs(k_dense2_output - manual_dense2_output)))


--- Verifying Dense_2 Layer ---
Dense_2 weights shape: (128, 64)
Dense_2 bias shape: (64,)
Dense_2 activation function: relu

Shape of Keras Dense_2 output: (1, 64)
Shape of manually calculated Dense_2 output: (1, 64)
Are Dense_2 outputs approximately equal? True
Max absolute difference for Dense_2: 2.9802322e-08


This block manually verifies the `dense_3` layer. It takes the output of `dense_2` as input, retrieves the layer’s weights, bias, and activation function, and computes the manual output by performing matrix multiplication of input and weights, adding the bias, and applying the activation (typically softmax for the final layer). The manual output is then compared with the Keras output using `np.allclose` and the maximum difference is printed, confirming the layer produces the expected results.

In [11]:
# Manual implementation and verification for DENSE_3 layer
print("\n--- Verifying Dense_3 Layer ---")

# Getting model parts relevant to dense_3 using helper
(start_model_dense3, layer_dense3, end_model_dense3) = get_reference_layer("dense_3", model)

# 'dense2output_keras_input_for_dense3' is the Keras output of the layer before dense_3 (i.e., dense_2)
dense2output_keras_input_for_dense3 = start_model_dense3(padded)

# Getting the Keras output for dense_3
k_dense3_output = layer_dense3(dense2output_keras_input_for_dense3)

# Converting the input to a numpy array for manual calculation
input_to_dense3_keras_output = dense2output_keras_input_for_dense3.numpy()

# Getting weights (kernel) and bias for dense_3
w1 = get_layer_weights("dense_3", model)[0]
b1 = get_layer_weights("dense_3", model)[1]

# Getting the activation function of the dense_3 layer
dense3_activation = model.get_layer("dense_3").activation
print(f"\nDense_3 activation function: {dense3_activation.__name__}")

# Manual calculation for dense_3 layer (pre-activation): input * weights + bias
manual_dense3_pre_activation = np.dot(input_to_dense3_keras_output, w1) + b1

# Apply the activation function (Softmax for the final output layer of this model)
if dense3_activation.__name__ == 'softmax':
    # Softmax for numerical stability
    exp_values = np.exp(manual_dense3_pre_activation - np.max(manual_dense3_pre_activation, axis=-1, keepdims=True))
    manual_dense3_output = exp_values / np.sum(exp_values, axis=-1, keepdims=True)
elif dense3_activation.__name__ == 'linear':
    manual_dense3_output = manual_dense3_pre_activation
else:
    print("Warning: Unhandled activation function. Assuming linear for comparison.")
    manual_dense3_output = manual_dense3_pre_activation

# Compare with Keras output for dense_3
print("\nShape of Keras Dense_3 output (k_dense3_output):", k_dense3_output.shape)
print("Shape of manually calculated Dense_3 output:", manual_dense3_output.shape)
print("Are outputs approximately equal?", np.allclose(k_dense3_output, manual_dense3_output, rtol=1e-5, atol=1e-8))
print("Max absolute difference:", np.max(np.abs(k_dense3_output - manual_dense3_output)))


--- Verifying Dense_3 Layer ---

Dense_3 activation function: softmax

Shape of Keras Dense_3 output (k_dense3_output): (1, 50281)
Shape of manually calculated Dense_3 output: (1, 50281)
Are outputs approximately equal? False
Max absolute difference: 1.1734664e-06


This block performs a full manual forward pass of the entire model, layer by layer, to verify that the integrated manual computation matches the Keras model output.

Starting with the padded input, it sequentially applies:

* **Embedding** – token IDs are mapped to vectors using the embedding matrix.
* **Conv1D** – a 1D convolution is computed manually with appropriate padding, strides, kernel multiplication, bias addition, and activation (ReLU).
* **GlobalMaxPooling1D** – the maximum value is taken across the sequence dimension for each feature.
* **Dense_2** – matrix multiplication with weights, bias addition, and activation (ReLU or softmax) is applied.
* **Dense_3** – the final output layer is computed similarly, typically with softmax activation.

Finally, the manually computed output is compared with the Keras model’s prediction using `np.allclose` and the maximum absolute difference, confirming that the manual integrated forward pass reproduces the model’s behavior exactly.

In [12]:
# Integrated Manual Model Forward Pass
print("\n--- Verifying Full Integrated Manual Model ---")

# Starting with the padded input
current_integrated_output = padded

# Layer 1: Embedding
# Getting weights directly for manual lookup
w_embedding1_integrated = get_layer_weights("embedding_1", model)[0]
current_integrated_output = w_embedding1_integrated[current_integrated_output]
print(f"After Embedding: {current_integrated_output.shape}")

# Layer 2: Conv1D
# Retrieving all necessary parameters from the Keras Conv1D layer
conv1d1_obj_integrated = model.get_layer("conv1d_1")
kernel_size_conv1d1_integrated = conv1d1_obj_integrated.kernel_size[0]
strides_conv1d1_integrated = conv1d1_obj_integrated.strides[0]
padding_conv1d1_integrated = conv1d1_obj_integrated.padding
activation_conv1d1_integrated = conv1d1_obj_integrated.activation

w_conv1d1_kernel_integrated, b_conv1d1_integrated = get_layer_weights("conv1d_1", model)

# Extracting input dimensions for manual convolution
batch_size_integrated, input_timesteps_integrated, input_dim_integrated = current_integrated_output.shape
filters_integrated = w_conv1d1_kernel_integrated.shape[2]

input_for_conv_integrated = current_integrated_output
output_timesteps_integrated = 0

# Handle 'same' padding for Conv1D, similar to the individual verification
if padding_conv1d1_integrated == 'valid':
    output_timesteps_integrated = (input_timesteps_integrated - kernel_size_conv1d1_integrated) // strides_conv1d1_integrated + 1
elif padding_conv1d1_integrated == 'same':
    output_timesteps_integrated = math.ceil(input_timesteps_integrated / strides_conv1d1_integrated)
    if (input_timesteps_integrated % strides_conv1d1_integrated == 0):
        pad_along_axis_integrated = max(kernel_size_conv1d1_integrated - strides_conv1d1_integrated, 0)
    else:
        pad_along_axis_integrated = max(kernel_size_conv1d1_integrated - (input_timesteps_integrated % strides_conv1d1_integrated), 0)
    pad_left_integrated = pad_along_axis_integrated // 2
    pad_right_integrated = pad_along_axis_integrated - pad_left_integrated
    input_for_conv_integrated = np.pad(current_integrated_output, ((0,0), (pad_left_integrated, pad_right_integrated), (0,0)), mode='constant')
    input_timesteps_integrated = input_for_conv_integrated.shape[1] # Update input timesteps for convolution loop
elif padding_conv1d1_integrated == 'causal':
    pad_left_integrated = kernel_size_conv1d1_integrated - 1
    pad_right_integrated = 0
    input_for_conv_integrated = np.pad(current_integrated_output, ((0,0), (pad_left_integrated, pad_right_integrated), (0,0)), mode='constant')
    input_timesteps_integrated = input_for_conv_integrated.shape[1]
    output_timesteps_integrated = (input_timesteps_integrated - kernel_size_conv1d1_integrated) // strides_conv1d1_integrated + 1
else:
    raise ValueError(f"Unsupported padding: {padding_conv1d1_integrated}")

integrated_conv1d1_pre_activation = np.zeros((batch_size_integrated, output_timesteps_integrated, filters_integrated))

for b in range(batch_size_integrated):
    for i in range(output_timesteps_integrated):
        start_idx_integrated = i * strides_conv1d1_integrated
        end_idx_integrated = start_idx_integrated + kernel_size_conv1d1_integrated
        input_slice_integrated = input_for_conv_integrated[b, start_idx_integrated:end_idx_integrated, :]
        for f in range(filters_integrated):
            integrated_conv1d1_pre_activation[b, i, f] = np.sum(input_slice_integrated * w_conv1d1_kernel_integrated[:, :, f]) + b_conv1d1_integrated[f]

current_integrated_output = integrated_conv1d1_pre_activation
if activation_conv1d1_integrated.__name__ == 'relu':
    current_integrated_output = np.maximum(0, current_integrated_output)
print(f"After Conv1D: {current_integrated_output.shape}")

# Layer 3: GlobalMaxPooling1D
# Apply global max pooling along the sequence dimension (axis=1)
current_integrated_output = np.max(current_integrated_output, axis=1)
print(f"After GlobalMaxPooling1D: {current_integrated_output.shape}")

# Layer 4: Dense_2
# Getting weights and bias for Dense_2
w_dense2_integrated, b_dense2_integrated = get_layer_weights("dense_2", model)
dense2_activation_integrated = model.get_layer("dense_2").activation

# Perform matrix multiplication and add bias
integrated_dense2_pre_activation = np.dot(current_integrated_output, w_dense2_integrated) + b_dense2_integrated
current_integrated_output = integrated_dense2_pre_activation
# Apply activation function
if dense2_activation_integrated.__name__ == 'softmax':
    exp_values_integrated = np.exp(integrated_dense2_pre_activation - np.max(integrated_dense2_pre_activation, axis=-1, keepdims=True))
    current_integrated_output = exp_values_integrated / np.sum(exp_values_integrated, axis=-1, keepdims=True)
elif dense2_activation_integrated.__name__ == 'relu':
    current_integrated_output = np.maximum(0, current_integrated_output)
print(f"After Dense_2: {current_integrated_output.shape}")

# Layer 5: Dense_3 (Final output layer)
# Getting weights and bias for Dense_3
w_dense3_integrated, b_dense3_integrated = get_layer_weights("dense_3", model)[0], get_layer_weights("dense_3", model)[1]
dense3_activation_integrated = model.get_layer("dense_3").activation

# Perform matrix multiplication and add bias
integrated_dense3_pre_activation = np.dot(current_integrated_output, w_dense3_integrated) + b_dense3_integrated
current_integrated_output = integrated_dense3_pre_activation
# Apply activation function (Softmax)
if dense3_activation_integrated.__name__ == 'softmax':
    exp_values_integrated = np.exp(integrated_dense3_pre_activation - np.max(integrated_dense3_pre_activation, axis=-1, keepdims=True))
    current_integrated_output = exp_values_integrated / np.sum(exp_values_integrated, axis=-1, keepdims=True)
print(f"After Dense_3 (final output): {current_integrated_output.shape}")


# Compare with original model.predict() output
# Get the full model's prediction for the same padded input
k_full_model_output = model.predict(padded, verbose=0)

print("\nShape of Keras Full Model output:", k_full_model_output.shape)
print("Shape of Integrated Manual Model output:", current_integrated_output.shape)
print("Are Integrated Manual Model outputs approximately equal?", np.allclose(k_full_model_output, current_integrated_output, rtol=1e-5, atol=1e-8))
print("Max absolute difference for Integrated Manual Model:", np.max(np.abs(k_full_model_output - current_integrated_output)))


--- Verifying Full Integrated Manual Model ---
After Embedding: (1, 30, 16)
After Conv1D: (1, 30, 128)
After GlobalMaxPooling1D: (1, 128)
After Dense_2: (1, 64)
After Dense_3 (final output): (1, 50281)

Shape of Keras Full Model output: (1, 50281)
Shape of Integrated Manual Model output: (1, 50281)
Are Integrated Manual Model outputs approximately equal? False
Max absolute difference for Integrated Manual Model: 1.1838629027927094e-06
